## Comprehension - Working with LLM APIs

### Import Libraries and Load Data

Import libraries, load the API keys, and setup the client

In [ ]:
# Install the LLM provider SDK/library of your choice (Gemini/Hugging Face/OpenAI) and load the API Key

# %pip install -q -U google-genai huggingface_hub python-dotenv openai

In [2]:
import openai
from google import genai
from huggingface_hub import InferenceClient

import os
from dotenv import load_dotenv

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
client = openai.Client(api_key=OPENAI_API_KEY)

Load the dataset

In [ ]:
# Load CSV into any type of data structure you prefer
import pandas as pd
import numpy as np
df = pd.read_csv('laptop_descriptions.csv')
df.head()

,laptop_description
0,The Dell Inspiron is a versatile laptop that c...
1,The MSI GL65 is a high-performance laptop desi...
2,The HP EliteBook is a premium laptop designed ...
3,The Lenovo IdeaPad is a versatile laptop that ...
4,The ASUS ZenBook Pro is a high-end laptop that...


---

## Task 1

Based on the dataset, classify the laptop into one of the following tags corresponding to their categories:
- general
- business
- gamer
- programmer
- multimedia

| Category | Description |
| --------------- | --------------- |
| general | For general purpose use such as light web browsing, editing documents etc. |
| business | For business users, the focus is on portability, battery backup and general purpose use. |
| gamer | For gamers, the focus is primarily on high-performance, a separate GPU for high-performance graphics, a high-end CPU processor etc. |
| programmer | For programmers, the focus is on performance, battery backup, high-end RAM etc. |
| multimedia | For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc. |


The final output dataframe should like the table below:

| laptop_description | category |
|--------------- |--------------- |
| The Dell Inspiron ... | **general** |
| The MSI GL65 ... | **gaming** |
|  ... | **...** |


### <font color='RED' >Question 1</font>

What is the laptop category of the laptop at the first index position (i.e. MSI GL65)? Use the following code to obtain the result: 

`df.iloc[1]['Category']`

Attempt this question through the 3 tasks given below.

#### **1.1** First, try creating a wrapper function which takes in a user request, and outputs one word - the category name.

This function will involve creating a system message to define the role, context, task, and output style. Note that you have to ensure one word outputs. This function will produce the output only for one laptop description, not the whole data.

Additionally, try using error handling as well.


You do not need to provide the category-related information or laptop data here. That will be given with user prompt.

In [ ]:
### Function to get LLM Response

def get_chat_response_q1(user_request):

  '''
  This function ONLY takes `user_request` as the input argument.
  As you can see, the System Prompt is given inside the function itself so we don't require to give it as an input argument
  '''
  
  # Define  model
  MODEL = 'gpt-4o-mini'

  # Default System Message
  SYSTEM_MESSAGE = '''
  You are a shopping assistant. The user will give you laptop description and some categories and their details. 
  You have to find out which of the categories does the laptop fit best according to description. 
  Remember to only give one word output, the category name, from the list of categories only, which resembles most closely.
  ''' 

  try:

    response = client.responses.create(
        model = MODEL,
        instructions=SYSTEM_MESSAGE,
        input = user_request
    )

    # Parse the response_content from the message
    response_content = response.output_text

    return response_content

  # Raise exception error
  except Exception as e:
    print(f"An error occurred: {e}")
    return None

#### **1.2** Prompt Definition

Now define the user prompt where you will give the description of laptops and the description of each category (use category descriptions from the table above). Include the exact task to be performed over these.

You can use f-string to set up a prompt which has a placeholder for the dataset (description of laptops). In the next part, we will attach the dataset to this f-string based prompt.

Note that a good way to provide the information about a laptop to the user prompt will be to iterate over the whole data and attach it to the placeholder row-by-row. Design your prompt keeping this in mind.

In [ ]:
# PROMPT DEFINITION

mcq1_prompt = '''
From the description of a laptop (delimited by '###'), you have to identify what role does the laptop serve. 
Refer to the key value pairs of categories and category details below. 
Identify which of the following details does the product description fits best and assign that category to that latpop. \n

Categories:
[
    'general': 'For general purpose use such as light web browsing, editing documents etc.'
    'business': 'For business users, the focus is on portability, battery backup and general purpose use.'
    'gamer': 'For gamers, the focus is primarily on high-performance, high-end graphics requirement, efficient processor etc.'
    'programmer': 'For programmers, the focus is on performance, battery backup, high-end RAM etc.'
    'multimedia': 'For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc.' # Write the prompt here
] \n ###
Laptop description: {description}
'''

# We have used string formatting to give description inputs along with the prompt.

#### **1.3** Tagging Laptops

Now, create a final wrapper function to combine the placeholder prompt with the dataset, and use the first wrapper function created in task 1.1 to assign a category to each laptop in the data. You can iterate over every laptop and assign it a category.

In [ ]:
# Code to tag the laptop based on their descriptions

def tag_laptop():

  laptop_df = df.copy()
  laptop_dict = laptop_df.to_dict(orient ='records')

  # Get the laptop_category for each laptop_decription df['laptop_description'] by iterating over the dataframe with a for-loop

  for i in range(len(laptop_dict)):

    prompt = mcq1_prompt.format(description=laptop_dict[i]['laptop_description'])

    laptop_category = get_chat_response_q1(prompt)
    
    # Assign the laptop category to the column laptop_category
    laptop_df.at[i,'Category'] = laptop_category

  # return the DataFrame
  return laptop_df

In [12]:
# Calling the function
tag_df = tag_laptop()

In [13]:
# Finding the category of a laptop at an index
tag_df.iloc[1]['Category']

'gamer'

**How many gamer laptops did you find?**

In [14]:
# Total number of products of each category
tag_df['Category'].value_counts()

Category
multimedia    6
general       5
gamer         5
business      4
Name: count, dtype: int64

---

## Task 2: Information Extraction


Extract the relevant values for the following dictionary items from the product description.
```
{
    "Brand": {"type": "string"},
    "Model Name": {"type": "string"},
    "GPU processor": {"type": "string"},
    "Display Resolution": {"type": "string"},
    "Weight": {"type": "string"},
    "Processor": {"type": "string"},
    "Clock speed": {"type": "string"},
    "Budget": {"type": "string"}
}
```

You can add this property dictionary for all the products to a list and output that list.

### <font color='RED' >Question 2</font>

Which of the following correctly represents the laptop tag values for the Laptop description at index position 10? The laptop is ASUS ROG Strix G.

Attempt this question through the 3 tasks below

---

Note: Most models like Gemini and GPT now use a Pydantic schema for structure validation. Pydantic is a library for data validation that enforces data structures, and coerces input data to predined data types.

The structure is usually defined using a `BaseModel` class. It has been already done for you below.

In [16]:
from pydantic import BaseModel

class Laptop(BaseModel):
    Brand: str
    Model_Name: str
    GPU_processor: str
    Display_Resolution: str
    Weight: str
    Processor: str
    Clock_speed: str
    Budget: str

This created schema can directly be passed with the model call which returns a Pydantic object adhering to it.

In [ ]:
# Gemini
response = client.models.generate_content(
    model=,
    contents=,
    config={
        "response_mime_type": "application/json",
        "response_schema": Laptop
    }
)

# OpenAI (.parse instead of .create)
response = client.responses.parse(
    model=,
    input=,
    text_format=Laptop,
)
response.output_parsed

#### **2.1** Prompt Definition

First define the user prompt where you will give the structure of the output, define the exact task, and again add the placeholder for laptop description using f-string.

You may want to use a fixed schema for structured output for this task. Though, you can first try using a simple prompt.

In [17]:
# Define user prompt

mcq2_prompt = '''
Laptop Decription: {description}
From the laptop decription above, you have to extract relevant values for the following dictionary items.
Try giving quantitative, absolute, or numerical outputs. Try not to give qualitative or adjective outputs. 
for example: If the processing speed of a laptop is 2.4GHz, then, in the "processing speed" key, give output as '2.4GHz' instead of 'very fast'.
Extract only one word values of these properties. Fill in the blanks for each product and output each product's dictionary in json format.
'''


# We are giving the product descriptions as well as the dictionary structure of our required output

#### **2.2** Function for Response Generation

Now, define a function that takes the Laptop Pydantic schema and generates an output for each laptop. Define system instructions and response type as well.

Similar to task 1.1, this will also work on singular entries of laptops

In [21]:
# Function to get response

def get_chat_response_q2(user_request):

  MODEL = 'gpt-4o-mini'# Define GPT model

  SYSTEM_MESSAGE = 'You are a helpful shopping assitant.'

  try:
    
    # Get the Response from the model
    response = openai.responses.parse(
        model = MODEL,
        input = user_request,
        instructions= SYSTEM_MESSAGE,
        text_format = Laptop
    )

    response_content = response.output_parsed
    # Parse the response_content from the message

    return response_content

  # Raise exception error
  except Exception as e:
    print(f"An error occurred: {e}")
    return None

#### **2.3** Extracting Data

Finally, create a final wrapper function to iterate over the laptop data like in task 1.3, provide the description in each row to the user query, and get the responses.

In [22]:
# Write the code to extract product information from the 'Description' value in the DataFrame
def extract_information():

  laptop_df = df.copy()
  laptop_dict = laptop_df.to_dict(orient ='records')

  # Creating an empty list to store the properties
  result = []

  # Get the relevant values for each of the property:
  for i in range(len(laptop_dict)):

    prompt = mcq2_prompt.format(description=laptop_dict[i]['laptop_description'])

    values = get_chat_response_q2(prompt)
    
    result.append(values)
    
    # We will print each of the dictionary of properties
    print(result[i])
  
  # But in the function output, we are returning the whole list as a whole.
  return result

  # Your solution can differ, but your end goal is to output the properties for all products in one single list.

In [23]:
# Calling the function
values = extract_information()

Brand='Dell' Model_Name='Inspiron' GPU_processor='Intel UHD' Display_Resolution='1920x1080' Weight='2.5kg' Processor='Intel Core i5' Clock_speed='2.4GHz' Budget='35000'
Brand='MSI' Model_Name='GL65' GPU_processor='NVIDIA GTX' Display_Resolution='1920x1080' Weight='2.3 kg' Processor='Intel Core i7' Clock_speed='2.6 GHz' Budget='55,000'
Brand='HP' Model_Name='EliteBook' GPU_processor='Intel UHD' Display_Resolution='1920x1080' Weight='1.5kg' Processor='Intel Core i7' Clock_speed='2.8GHz' Budget='90000'
Brand='Lenovo' Model_Name='IdeaPad' GPU_processor='Intel UHD' Display_Resolution='1366x768' Weight='2.2kg' Processor='Intel Core i3' Clock_speed='2.1GHz' Budget='25000'
Brand='ASUS' Model_Name='ZenBook Pro' GPU_processor='NVIDIA RTX' Display_Resolution='3840x2160' Weight='1.8 kg' Processor='Intel Core i9' Clock_speed='3.1 GHz' Budget='200,000'
Brand='Acer' Model_Name='Predator' GPU_processor='GTX' Display_Resolution='1920x1080' Weight='3.2kg' Processor='i7' Clock_speed='2.8GHz' Budget='8000

In [24]:
values

[Laptop(Brand='Dell', Model_Name='Inspiron', GPU_processor='Intel UHD', Display_Resolution='1920x1080', Weight='2.5kg', Processor='Intel Core i5', Clock_speed='2.4GHz', Budget='35000'),
 Laptop(Brand='MSI', Model_Name='GL65', GPU_processor='NVIDIA GTX', Display_Resolution='1920x1080', Weight='2.3 kg', Processor='Intel Core i7', Clock_speed='2.6 GHz', Budget='55,000'),
 Laptop(Brand='HP', Model_Name='EliteBook', GPU_processor='Intel UHD', Display_Resolution='1920x1080', Weight='1.5kg', Processor='Intel Core i7', Clock_speed='2.8GHz', Budget='90000'),
 Laptop(Brand='Lenovo', Model_Name='IdeaPad', GPU_processor='Intel UHD', Display_Resolution='1366x768', Weight='2.2kg', Processor='Intel Core i3', Clock_speed='2.1GHz', Budget='25000'),
 Laptop(Brand='ASUS', Model_Name='ZenBook Pro', GPU_processor='NVIDIA RTX', Display_Resolution='3840x2160', Weight='1.8 kg', Processor='Intel Core i9', Clock_speed='3.1 GHz', Budget='200,000'),
 Laptop(Brand='Acer', Model_Name='Predator', GPU_processor='GTX'

We can use these extracted properties to build a more refined shopping assistant. Try doing that on your own.


Apart from Pydantic classes, you can also define JSON style schema for both Gemini and OpenAI.

In [ ]:
# Gemini

laptop_schema = {
    "type": "object",
    "properties": {
        "Brand": {"type": "string"},
        "Model Name": {"type": "string"},
        "GPU processor": {"type": "string"},
        "Display Resolution": {"type": "string"},
        "Weight": {"type": "string"},
        "Processor": {"type": "string"},
        "Clock speed": {"type": "string"},
        "Budget": {"type": "string"}
    },
    "required": [
        "Brand",
        "Model Name",
        "GPU processor",
        "Display Resolution",
        "Weight",
        "Processor",
        "Clock speed",
        "Budget"
    ]
}

response = client.models.generate_content(
    model=,
    contents=,
    config={
        "response_mime_type": "application/json",
        "response_schema": laptop_schema
    }
)

print(response.text)

In [ ]:
# OpenAI

schema = {
        "format": {
            "type": "json_schema",
            "name": "Laptop",
            "schema": {
                "type": "object",
                "properties": {
                    "Brand": {"type": "string"},
                    "Model Name": {"type": "string"},
                    "GPU processor": {"type": "string"},
                    "Display Resolution": {"type": "string"},
                    "Weight": {"type": "string"},
                    "Processor": {"type": "string"},
                    "Clock speed": {"type": "string"},
                    "Budget": {"type": "string"}
                },
                "required": [
                    "Brand",
                    "Model Name",
                    "GPU processor",
                    "Display Resolution",
                    "Weight",
                    "Processor",
                    "Clock speed",
                    "Budget"
                ],
                "additionalProperties": False
            },
            "strict": True
        }
}


response = client.responses.create(
model=,
input=,
text= schema
)

print(response.output_text)

The outputs from these will resemble JSON objects more. Try using these for data extraction.